# VisionAI Inspector — Stage 3: Deep Learning Anomaly Detection (PyTorch)

**Goal:** build two deep-learning approaches to the same anomaly-detection
problem Stage 2 tackled with classical ML, and compare all of them fairly
on the exact same untouched test set.

**Protocol (unchanged from Stage 1/2):** train only on normal (`good`)
images; never train or tune on official test images; calibrate any decision
threshold from validation scores only.

**Two models, deliberately different mechanisms — matching the project's
"custom vs pretrained" comparison goal:**

- **Model A — Custom Convolutional Autoencoder.** Trained from scratch on
  normal images only. Learns to compress and reconstruct normal bottles;
  anomalies reconstruct poorly. This is where you'll actually touch
  convolution, pooling, activations, dropout, backprop, epochs/batches,
  and learning rate — the PyTorch fundamentals.
- **Model B — Pretrained ResNet18 (transfer learning), frozen.** No
  training of the backbone at all — we reuse ImageNet-learned visual
  features and feed them into the same classical anomaly detectors from
  Stage 2 (One-Class SVM, Isolation Forest), just with richer features.

Like Stage 1/2, this notebook auto-detects real vs. synthetic placeholder
data and tags every such output accordingly.


## 1. Setup, imports, and loading Stage 1's splits

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_utils import build_protocol_splits, make_synthetic_placeholder_dataset
from torch_dataset import BottleDataset
from autoencoder import ConvAutoencoder
from pretrained_features import build_resnet_feature_extractor, extract_features
from evaluation import calibrate_threshold, evaluate_anomaly_scores

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "mvtec_ad"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CATEGORY = "bottle"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

category_path = RAW_ROOT / CATEGORY
USING_SYNTHETIC_DATA = not category_path.exists()
if USING_SYNTHETIC_DATA:
    print("Real MVTec data not found — generating synthetic placeholder for pipeline testing.")
    make_synthetic_placeholder_dataset(RAW_ROOT, category=CATEGORY, seed=RANDOM_SEED)

DATA_TAG = "[SYNTHETIC PLACEHOLDER DATA] " if USING_SYNTHETIC_DATA else ""

train_csv = PROCESSED_ROOT / f"{CATEGORY}_train.csv"
val_csv = PROCESSED_ROOT / f"{CATEGORY}_val.csv"
test_csv = PROCESSED_ROOT / f"{CATEGORY}_test.csv"

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)
else:
    print("Stage 1 CSVs not found — regenerating splits with the same protocol-preserving logic.")
    train_df, val_df, test_df = build_protocol_splits(category_path, val_frac=0.15, seed=RANDOM_SEED)

y_test_true = (test_df["label"] == "defective").astype(int).to_numpy()

print(f"\n{DATA_TAG}train (normal only): {len(train_df)}")
print(f"{DATA_TAG}val (normal only):   {len(val_df)}")
print(f"{DATA_TAG}test (untouched):    {len(test_df)}  -- label counts: {test_df['label'].value_counts().to_dict()}")

## 2. PyTorch Datasets and DataLoaders (for the autoencoder)

`BottleDataset` (see `src/torch_dataset.py`) wraps our metadata table so a
`DataLoader` can fetch it in shuffled batches. For the autoencoder we use
small 64x64 images scaled to `[0, 1]` — `transforms.ToTensor()` handles
both the resize-to-tensor conversion and the 0-1 scaling in one step, and
also rearranges the image into PyTorch's expected `(channels, height,
width)` tensor layout.

`DataLoader`'s `batch_size` controls how many images are processed together
in one forward/backward pass — a bigger batch gives a more stable gradient
estimate per step but uses more memory; `shuffle=True` on the training
loader means the model sees images in a different order each epoch, which
prevents it from learning something spurious about ordering.


In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 16

ae_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # converts to (C, H, W) tensor, scaled to [0, 1]
])

train_dataset_ae = BottleDataset(train_df, transform=ae_transform)
val_dataset_ae = BottleDataset(val_df, transform=ae_transform)
test_dataset_ae = BottleDataset(test_df, transform=ae_transform)

train_loader_ae = DataLoader(train_dataset_ae, batch_size=BATCH_SIZE, shuffle=True)
val_loader_ae = DataLoader(val_dataset_ae, batch_size=BATCH_SIZE, shuffle=False)
test_loader_ae = DataLoader(test_dataset_ae, batch_size=BATCH_SIZE, shuffle=False)

sample_batch, sample_labels = next(iter(train_loader_ae))
print(f"{DATA_TAG}One batch shape: {sample_batch.shape}  (batch, channels, height, width)")
print(f"{DATA_TAG}Pixel value range in batch: [{sample_batch.min():.3f}, {sample_batch.max():.3f}]")

## 3. Model A — Custom Convolutional Autoencoder

See `src/autoencoder.py` for the full architecture with every layer
explained. Quick summary of the shape as data flows through:

`64x64x3 -> (conv+pool)x3 -> 8x8x64 -> flatten+linear -> latent vector (64) -> linear+reshape -> (upsample+conv)x3 -> 64x64x3`

The 64-number latent vector in the middle is the bottleneck: the entire
image has to be compressible into just those 64 numbers for the decoder to
rebuild it. That's what forces the network to learn efficient, general
structure rather than memorizing pixels.


In [ ]:
LATENT_DIM = 64
autoencoder = ConvAutoencoder(latent_dim=LATENT_DIM, dropout_p=0.1).to(DEVICE)

n_params = sum(p.numel() for p in autoencoder.parameters())
print(f"{DATA_TAG}Autoencoder parameter count: {n_params:,}")

# Sanity-check the forward pass shape before training anything.
with torch.no_grad():
    test_out = autoencoder(sample_batch.to(DEVICE))
print(f"{DATA_TAG}Input shape: {tuple(sample_batch.shape)}  ->  Output shape: {tuple(test_out.shape)}")
assert test_out.shape == sample_batch.shape, "Autoencoder output shape must match input shape!"


## 4. Training loop (written by hand)

This is the actual mechanics of neural network training:

1. **Forward pass** — run a batch through the model to get reconstructions.
2. **Loss** — Mean Squared Error between input and reconstruction: how
   different is the output from the input, pixel by pixel? This is the one
   number the entire training process tries to minimize.
3. **Backward pass** (`loss.backward()`) — PyTorch's autograd computes the
   gradient of the loss with respect to every parameter in the network:
   for each weight, "if I nudge this slightly, does the loss go up or
   down, and by how much?"
4. **Optimizer step** (`optimizer.step()`) — Adam uses those gradients to
   actually update every weight, nudging each one in the direction that
   reduces the loss. The **learning rate** controls how big that nudge is —
   too high and training can become unstable/diverge, too low and it takes
   forever to improve.
5. **Epoch** — one full pass through the entire training set. We track
   both train and validation loss after every epoch — if train loss keeps
   dropping while validation loss stops improving or rises, that's
   **overfitting**: the model is memorizing training-specific detail
   rather than learning general "what does normal look like" structure.
   Dropout (already in the architecture) is one direct defense against
   this.


In [ ]:
N_EPOCHS = 30
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

train_losses, val_losses = [], []

for epoch in range(1, N_EPOCHS + 1):
    autoencoder.train()
    running_train_loss = 0.0
    for images, _labels in train_loader_ae:
        images = images.to(DEVICE)

        optimizer.zero_grad()
        reconstructions = autoencoder(images)
        loss = loss_fn(reconstructions, images)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)
    train_loss = running_train_loss / len(train_dataset_ae)

    autoencoder.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, _labels in val_loader_ae:
            images = images.to(DEVICE)
            reconstructions = autoencoder(images)
            loss = loss_fn(reconstructions, images)
            running_val_loss += loss.item() * images.size(0)
    val_loss = running_val_loss / len(val_dataset_ae)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"{DATA_TAG}Epoch {epoch:3d}/{N_EPOCHS}  train_loss={train_loss:.5f}  val_loss={val_loss:.5f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.xlabel("epoch")
plt.ylabel("MSE reconstruction loss")
plt.title(f"{DATA_TAG}Autoencoder training curve")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Visualize reconstructions

The real test of "did this work": can you *see* the model doing a better job reconstructing normal images than defective ones? This is the visual intuition behind why reconstruction error works as an anomaly score.

In [ ]:
def show_reconstructions(dataset, indices, title):
    fig, axes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 6))
    autoencoder.eval()
    with torch.no_grad():
        for col, idx in enumerate(indices):
            img, label = dataset[idx]
            recon = autoencoder(img.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)

            axes[0, col].imshow(img.permute(1, 2, 0).numpy())
            axes[0, col].set_title("original" + (" (defective)" if label == 1 else " (good)"))
            axes[0, col].axis("off")

            axes[1, col].imshow(recon.permute(1, 2, 0).numpy())
            axes[1, col].set_title("reconstructed")
            axes[1, col].axis("off")
    fig.suptitle(f"{DATA_TAG}{title}", y=1.05)
    plt.tight_layout()
    plt.show()

good_indices = [i for i in range(len(test_dataset_ae)) if test_dataset_ae.labels[i] == 0][:3]
defective_indices = [i for i in range(len(test_dataset_ae)) if test_dataset_ae.labels[i] == 1][:3]

show_reconstructions(test_dataset_ae, good_indices, "Normal test images: original vs reconstructed")
show_reconstructions(test_dataset_ae, defective_indices, "Defective test images: original vs reconstructed")

## 6. Reconstruction error as an anomaly score

For each image, the anomaly score is its per-image MSE between input and
reconstruction — a single number instead of MSE averaged across a whole
batch. Higher error = more anomalous (opposite convention from Stage 2's
OCSVM/Isolation Forest `decision_function`, where higher = more normal —
`src/evaluation.py`'s `higher_is_anomalous` flag handles this explicitly so
we don't mix them up).

Threshold calibration follows the exact same rule as Stage 2: computed only
from validation (normal-only) reconstruction errors, never from test
labels.


In [ ]:
def per_image_mse(loader):
    autoencoder.eval()
    errors = []
    with torch.no_grad():
        for images, _labels in loader:
            images = images.to(DEVICE)
            recon = autoencoder(images)
            # mean over channel/height/width, keep per-image
            batch_errors = ((recon - images) ** 2).mean(dim=[1, 2, 3])
            errors.append(batch_errors.cpu().numpy())
    return np.concatenate(errors)

val_errors_ae = per_image_mse(val_loader_ae)
test_errors_ae = per_image_mse(test_loader_ae)

THRESHOLD_PERCENTILE = 5
ae_threshold = calibrate_threshold(val_errors_ae, THRESHOLD_PERCENTILE, higher_is_anomalous=True)
print(f"{DATA_TAG}Autoencoder threshold (from val, {100-THRESHOLD_PERCENTILE}th percentile of normal errors): {ae_threshold:.6f}")

ae_result = evaluate_anomaly_scores(
    "Custom Autoencoder", y_test_true, test_errors_ae, ae_threshold, higher_is_anomalous=True, tag=DATA_TAG
)

## 7. Model B — Pretrained ResNet18 (transfer learning) + classical anomaly detectors

See `src/pretrained_features.py` for the full reasoning. In short: we load
a ResNet18 already trained on ImageNet, strip its final classification
layer, freeze every weight, and use it purely as a fixed feature extractor
— each image becomes a 512-number vector summarizing what the network
"sees" in it. We never update the backbone's weights here.

Those 512-dim vectors then go through the **exact same classical anomaly
detectors from Stage 2** (One-Class SVM, Isolation Forest) — same code
path, same hyperparameters, same evaluation function. The only thing that
changes is the features. That's what makes this a fair test of "does a
pretrained deep feature representation beat hand-crafted features?"


In [ ]:
RESNET_IMG_SIZE = 224  # standard ImageNet input size these weights expect

resnet, weights_are_pretrained, resnet_preprocess = build_resnet_feature_extractor(DEVICE)
PRETRAINED_TAG = "" if weights_are_pretrained else "[RANDOM-INIT FALLBACK, NOT TRANSFER LEARNING] "

train_dataset_resnet = BottleDataset(train_df, transform=resnet_preprocess)
val_dataset_resnet = BottleDataset(val_df, transform=resnet_preprocess)
test_dataset_resnet = BottleDataset(test_df, transform=resnet_preprocess)

train_loader_resnet = DataLoader(train_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False)
val_loader_resnet = DataLoader(val_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False)
test_loader_resnet = DataLoader(test_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False)

print(f"{PRETRAINED_TAG}{DATA_TAG}Extracting ResNet18 features for train/val/test...")
X_train_resnet = extract_features(resnet, train_loader_resnet, DEVICE)
X_val_resnet = extract_features(resnet, val_loader_resnet, DEVICE)
X_test_resnet = extract_features(resnet, test_loader_resnet, DEVICE)

print(f"{PRETRAINED_TAG}{DATA_TAG}X_train_resnet shape: {X_train_resnet.shape}")

## 8. Fit Stage 2's classical anomaly detectors on the new (ResNet) features

Scaling fit on train only, thresholds calibrated on val only — identical discipline to Stage 2, now applied to a richer feature representation.

In [ ]:
resnet_scaler = StandardScaler()
X_train_resnet_scaled = resnet_scaler.fit_transform(X_train_resnet)
X_val_resnet_scaled = resnet_scaler.transform(X_val_resnet)
X_test_resnet_scaled = resnet_scaler.transform(X_test_resnet)

ocsvm_resnet = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm_resnet.fit(X_train_resnet_scaled)
val_scores_ocsvm_resnet = ocsvm_resnet.decision_function(X_val_resnet_scaled)
test_scores_ocsvm_resnet = ocsvm_resnet.decision_function(X_test_resnet_scaled)
ocsvm_resnet_threshold = calibrate_threshold(val_scores_ocsvm_resnet, THRESHOLD_PERCENTILE, higher_is_anomalous=False)
ocsvm_resnet_result = evaluate_anomaly_scores(
    "One-Class SVM + ResNet18 features", y_test_true, test_scores_ocsvm_resnet, ocsvm_resnet_threshold,
    higher_is_anomalous=False, tag=PRETRAINED_TAG + DATA_TAG
)

print()
iso_resnet = IsolationForest(n_estimators=200, contamination=0.01, random_state=RANDOM_SEED)
iso_resnet.fit(X_train_resnet_scaled)
val_scores_iso_resnet = iso_resnet.decision_function(X_val_resnet_scaled)
test_scores_iso_resnet = iso_resnet.decision_function(X_test_resnet_scaled)
iso_resnet_threshold = calibrate_threshold(val_scores_iso_resnet, THRESHOLD_PERCENTILE, higher_is_anomalous=False)
iso_resnet_result = evaluate_anomaly_scores(
    "Isolation Forest + ResNet18 features", y_test_true, test_scores_iso_resnet, iso_resnet_threshold,
    higher_is_anomalous=False, tag=PRETRAINED_TAG + DATA_TAG
)

## 9. Full comparison: Stage 2 baseline vs. both Stage 3 approaches

Loads Stage 2's saved summary (if available) and lines it up against both
Stage 3 models — hand-crafted features, learned-from-scratch features
(autoencoder), and transfer-learned features, all evaluated identically on
the same untouched test set.


In [ ]:
rows = [
    {"model": "Autoencoder (Stage 3, from scratch)", "roc_auc": ae_result["auc"]},
    {"model": f"{PRETRAINED_TAG}One-Class SVM + ResNet18 (Stage 3, transfer learning)", "roc_auc": ocsvm_resnet_result["auc"]},
    {"model": f"{PRETRAINED_TAG}Isolation Forest + ResNet18 (Stage 3, transfer learning)", "roc_auc": iso_resnet_result["auc"]},
]

stage2_summary_path = PROCESSED_ROOT / f"{CATEGORY}_stage2_baseline_summary.csv"
if stage2_summary_path.exists():
    stage2_summary = pd.read_csv(stage2_summary_path)
    for _, row in stage2_summary.iterrows():
        rows.append({"model": f"{row['model']} (Stage 2, hand-crafted features)", "roc_auc": row["roc_auc"]})
else:
    print("Stage 2 summary CSV not found -- run Stage 2's notebook first to include it in this comparison.")

comparison_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
print(f"{DATA_TAG}")
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(comparison_df["model"], comparison_df["roc_auc"], color="#4C72B0")
ax.set_xlabel("ROC-AUC (test set)")
ax.set_title(f"{DATA_TAG}Stage 2 vs Stage 3 — anomaly detection comparison")
ax.set_xlim(0, 1)
fig.tight_layout(pad=2.0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))

fpr_ae, tpr_ae, _ = roc_curve(y_test_true, test_errors_ae)
fpr_ocsvm_r, tpr_ocsvm_r, _ = roc_curve(y_test_true, -test_scores_ocsvm_resnet)
fpr_iso_r, tpr_iso_r, _ = roc_curve(y_test_true, -test_scores_iso_resnet)

ax.plot(fpr_ae, tpr_ae, label=f"Autoencoder (AUC={ae_result['auc']:.3f})")
ax.plot(fpr_ocsvm_r, tpr_ocsvm_r, label=f"OCSVM+ResNet18 (AUC={ocsvm_resnet_result['auc']:.3f})")
ax.plot(fpr_iso_r, tpr_iso_r, label=f"IsoForest+ResNet18 (AUC={iso_resnet_result['auc']:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="random chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"{DATA_TAG}Stage 3 ROC curves")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
short_tag = "(fallback) " if not weights_are_pretrained else ""

ConfusionMatrixDisplay.from_predictions(y_test_true, ae_result["y_pred"], display_labels=["good", "defective"], ax=axes[0], colorbar=False)
axes[0].set_title(f"{DATA_TAG}Autoencoder")

ConfusionMatrixDisplay.from_predictions(y_test_true, ocsvm_resnet_result["y_pred"], display_labels=["good", "defective"], ax=axes[1], colorbar=False)
axes[1].set_title(f"{DATA_TAG}{short_tag}OCSVM + ResNet18")

ConfusionMatrixDisplay.from_predictions(y_test_true, iso_resnet_result["y_pred"], display_labels=["good", "defective"], ax=axes[2], colorbar=False)
axes[2].set_title(f"{DATA_TAG}{short_tag}IsoForest + ResNet18")

fig.tight_layout(pad=2.0)
plt.show()

## 10. Save Stage 3 outputs

In [ ]:
results_df = test_df[["filepath", "label", "defect_type"]].copy()
results_df["autoencoder_error"] = test_errors_ae
results_df["autoencoder_pred_defective"] = ae_result["y_pred"]
results_df["ocsvm_resnet_score"] = test_scores_ocsvm_resnet
results_df["ocsvm_resnet_pred_defective"] = ocsvm_resnet_result["y_pred"]
results_df["isoforest_resnet_score"] = test_scores_iso_resnet
results_df["isoforest_resnet_pred_defective"] = iso_resnet_result["y_pred"]

out_path = PROCESSED_ROOT / f"{CATEGORY}_stage3_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows -> {out_path}")

comparison_path = PROCESSED_ROOT / f"{CATEGORY}_stage3_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"Saved comparison table -> {comparison_path}")

torch.save(autoencoder.state_dict(), PROCESSED_ROOT / f"{CATEGORY}_autoencoder_weights.pt")
print(f"Saved trained autoencoder weights -> {PROCESSED_ROOT / f'{CATEGORY}_autoencoder_weights.pt'}")

print(f"\n{'SYNTHETIC PLACEHOLDER outputs saved.' if USING_SYNTHETIC_DATA else 'Real dataset outputs saved.'}")
if not weights_are_pretrained:
    print("NOTE: ResNet18 results used RANDOM-INIT fallback weights, not real ImageNet pretraining --")
    print("      these are a pipeline smoke-test only, not a genuine transfer-learning result.")

## 11. Summary — Stage 3 complete

What we built:

- A hand-built PyTorch `Dataset`/`DataLoader` pipeline (`src/torch_dataset.py`).
- **Model A**: a custom convolutional autoencoder trained from scratch on
  normal-only images, with a fully manual training loop — covering
  convolution, pooling, activations, dropout, backprop, epochs/batches,
  and learning rate.
- **Model B**: a frozen, pretrained ResNet18 used purely as a feature
  extractor (genuine transfer learning — no backbone fine-tuning), feeding
  the exact same classical anomaly detectors from Stage 2.
- Identical evaluation discipline throughout: thresholds calibrated from
  validation (normal-only) scores only, test labels touched exactly once,
  at the end, for scoring.
- A full comparison across Stage 2 (hand-crafted features) and both Stage 3
  approaches, on the same untouched test set.

**Important caveat if you're reading real results:** if the notebook
printed a `RANDOM-INIT FALLBACK` warning for Model B, your environment
couldn't reach `download.pytorch.org` to fetch real ImageNet weights —
check your internet connection and re-run; the random-init results are not
a real transfer-learning comparison.

**Next (Stage 4):** extend beyond whole-image anomaly detection into
object detection and segmentation — using MVTec's `ground_truth` masks
(which we haven't touched yet) to localize *where* a defect is, not just
whether one is present.
